# Gold · KPI Detalle por Región

## Propósito

Calcular los indicadores a nivel de región y modo de envío, y entregarlos como snapshot periódico.

## Flujo

1. Calcular los indicadores por región y modo de envío a partir del resumen por pedido.
2. Guardar el resultado como snapshot del corte, marcando el más reciente.

## Destino

Capa Gold.

**Tabla:** `prod_dataco.gld_dataco.kpi_detalle_region`

**Grano:** región y modo de envío, por corte

## Indicadores

Los mismos del Resumen Ejecutivo: On-Time Delivery %, Late Delivery Risk %, Shipping Variance, Profit Margin %, Revenue por cliente, Beneficio por orden y Lead Time promedio.

## Historial

* `fecha_snapshot` corresponde a la fecha de fin del corte.
* `es_ultimo_snapshot` marca el snapshot más reciente.
* El proceso es idempotente por corte.

## Resultado esperado

El detalle por región listo para la pestaña Detalle Región del Excel.

In [0]:
# ============================================================
# GLD_KPI_DETALLE_REGION — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog", "prod_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "gld_dataco", "Esquema Gold")

gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

GOLD_ORDER_SUMMARY_FQN = f"{gold_catalog}.{gold_schema}.order_summary"
GOLD_KPI_DETALLE_REGION_FQN = f"{gold_catalog}.{gold_schema}.kpi_detalle_region"

print(f"✓ Origen: {GOLD_ORDER_SUMMARY_FQN}")
print(f"✓ Destino: {GOLD_KPI_DETALLE_REGION_FQN}")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_REGION — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control", key="id_ejecucion", default="", debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_kpi_detalle_region"
CAPA = "gold"

spark.sql(f"""
INSERT INTO prod_dataco.metadata.etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_REGION — Celda 3: Función de agregación de los 7 KPIs
# ============================================================

from pyspark.sql import functions as F

df_base = spark.table(GOLD_ORDER_SUMMARY_FQN)

def calcular_kpis(df, agrupar_por=None):
    agg_exprs = [
        F.avg(F.when(~F.col("es_excluido_otd_leadtime"), F.col("es_entrega_a_tiempo"))).alias("on_time_delivery_pct"),
        F.avg(F.col("late_delivery_risk_calculado")).alias("late_delivery_risk_pct"),
        F.avg(F.when(~F.col("es_excluido_otd_leadtime"), F.col("shipping_variance"))).alias("shipping_variance_avg"),
        (F.sum("total_profit") / F.nullif(F.sum("total_sales"), F.lit(0)) * 100).alias("profit_margin_pct"),
        (F.sum("total_sales") / F.nullif(F.countDistinct("cliente_key"), F.lit(0))).alias("revenue_por_cliente"),
        F.avg("avg_benefit_per_order").alias("beneficio_por_orden"),
        F.avg(F.when(~F.col("es_excluido_otd_leadtime"), F.col("days_for_shipping_real"))).alias("lead_time_promedio"),
        F.sum(F.col("es_sospechoso_fraude").cast("int")).alias("total_sospechosos_fraude"),
        F.count("*").alias("total_pedidos")
    ]
    if agrupar_por:
        return df.groupBy(agrupar_por).agg(*agg_exprs)
    else:
        return df.agg(*agg_exprs)

print("✓ Función de agregación definida")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_REGION — Celda 4: Agrupar por región + modo de envío
# ============================================================

df_kpi_detalle_region = calcular_kpis(df_base, agrupar_por=["order_region", "shipping_mode"])

print(f"✓ Filas calculadas: {df_kpi_detalle_region.count()}")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_REGION — Celda 5: Snapshot periódico (historial)
# ============================================================

from delta.tables import DeltaTable

corte_actual = df_base.select(F.max("fecha_fin_corte").alias("m")).collect()[0]["m"]

df_kpi_detalle_region_final = df_kpi_detalle_region.withColumn("fecha_snapshot", F.lit(corte_actual))

if spark.catalog.tableExists(GOLD_KPI_DETALLE_REGION_FQN):
    ya_existe = (
        spark.table(GOLD_KPI_DETALLE_REGION_FQN)
        .filter(F.col("fecha_snapshot") == corte_actual)
        .limit(1).count() > 0
    )

    if not ya_existe:
        delta_tbl = DeltaTable.forName(spark, GOLD_KPI_DETALLE_REGION_FQN)
        delta_tbl.update(
            condition=F.col("es_ultimo_snapshot") == True,
            set={"es_ultimo_snapshot": F.lit(False)}
        )
        (
            df_kpi_detalle_region_final.withColumn("es_ultimo_snapshot", F.lit(True))
            .write.format("delta").mode("append").saveAsTable(GOLD_KPI_DETALLE_REGION_FQN)
        )
        print(f"✓ Nuevo snapshot agregado para el corte {corte_actual}")
    else:
        print(f"⚠️ El snapshot del corte {corte_actual} ya existía — no se duplicó (idempotente)")
else:
    (
        df_kpi_detalle_region_final.withColumn("es_ultimo_snapshot", F.lit(True))
        .write.format("delta").mode("overwrite").saveAsTable(GOLD_KPI_DETALLE_REGION_FQN)
    )
    print(f"✓ Tabla {GOLD_KPI_DETALLE_REGION_FQN} creada con el primer snapshot")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_REGION — Celda 6: ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE prod_dataco.metadata.etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")